# machine-learning_013

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (50).ipynb`

| Field | Value |
|---|---|
| Section | `machine_learning` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `daily` |
| Code cells | 11 |
| Detected functions | calculate_indicators, calculate_indicators, apply_trading_strategy, train_ml_model, live_trade_decision, backtest_and_live_trade, calculate_backtest_metrics, calculate_indicators, backtest_single_ticker, backtest_strategy, run_backtest_for_params |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/filtered_data.csv')
data

In [ ]:
data = data[data['Ticker'] == 'ICICIBANK']
data

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm  # For progress bar

# ===== Helper Functions for Indicator Calculations =====
def calculate_indicators(data):
    """
    Calculate VWAP, RSI, and Stochastic Oscillator for the given data.
    """
    # VWAP Calculation
    data['Typical Price'] = (data['high'] + data['low'] + data['close']) / 3
    data['Cumulative Vol * Price'] = (data['volume'] * data['Typical Price']).cumsum()
    data['Cumulative Volume'] = data['volume'].cumsum()
    data['VWAP'] = data['Cumulative Vol * Price'] / data['Cumulative Volume']

    # RSI Calculation
    delta = data['close'].diff()
    gain = delta.clip(lower=0).rolling(window=31).mean()
    loss = -delta.clip(upper=0).rolling(window=31).mean()
    data['RSI'] = 100 - (100 / (1 + gain / loss))

    # Stochastic Oscillator
    data['Low_Min'] = data['low'].rolling(window=8).min()
    data['High_Max'] = data['high'].rolling(window=8).max()
    data['%K'] = ((data['close'] - data['Low_Min']) / (data['High_Max'] - data['Low_Min'])) * 100
    data['%K'] = data['%K'].rolling(window=7).mean()  # Smoothing %K
    data['%D'] = data['%K'].rolling(window=3).mean()  # Smoothing %D

# ===== Strategy Parameters =====
capital = 100000  # Capital to trade per position
stop_loss_pct = 0.02  # 1% stop loss
take_profit_pct = 0.02  # 2.5% take profit
tradebook = []  # To store trade details

# ===== Trading Logic =====
data['date'] = pd.to_datetime(data['date'])  # Ensure 'date' is in datetime format
end_time = pd.to_datetime("15:25").time()
cutoff_time = pd.to_datetime("11:30").time()

# Iterate through unique tickers
for ticker in tqdm(data['Ticker'].unique(), desc="Running Backtest", ncols=100):
    ticker_data = data[data['Ticker'] == ticker].copy()

    # Apply indicator calculations to the ticker data
    calculate_indicators(ticker_data)

    # Drop rows with NaN due to rolling calculations
    #ticker_data.dropna(subset=['VWAP', 'RSI', '%K', '%D'], inplace=True)

    position = 0  # Track current position (0 = no position, > 0 = long, < 0 = short)
    entry_price = 0
    entry_time = None
    cumulative_pnl = 0  # Cumulative PnL for the ticker

    # Trading Simulation for the current ticker
    for i, row in ticker_data.iterrows():

        # Skip excluded day (default: Friday)
        if row['date'].weekday() == 4:
                continue

        # Trading Conditions
        buy_condition = (row['RSI'] < 35) and (row['%K'] < 20) and (row['close'] > row['VWAP'])
        short_condition = (row['RSI'] > 70) and (row['%K'] > 70) and (row['close'] < row['VWAP'])

        # ===== Entry Logic =====
        if position == 0:
            if buy_condition and row['date'].time() <= cutoff_time:
                position = capital / row['close']
                entry_price = row['close']
                entry_time = row['date']
                tradebook.append({
                    'Ticker': ticker, 'Entry Time': entry_time, 'Entry Price': entry_price,
                    'Exit Time': None, 'Exit Price': None, 'PNL': None, 'Exit Reason': None,
                    'Cumulative PNL': cumulative_pnl, 'Position': 'Long'
                })
            elif short_condition and row['date'].time() <= cutoff_time:
                position = -capital / row['close']
                entry_price = row['close']
                entry_time = row['date']
                tradebook.append({
                    'Ticker': ticker, 'Entry Time': entry_time, 'Entry Price': entry_price,
                    'Exit Time': None, 'Exit Price': None, 'PNL': None, 'Exit Reason': None,
                    'Cumulative PNL': cumulative_pnl, 'Position': 'Short'
                })

        # ===== Exit Logic =====
        if position != 0:
            exit_price = row['close']
            pnl = (exit_price - entry_price) * position - 0.001 * capital  # Deduct brokerage

            if position > 0:  # Long Exit
                take_profit = entry_price * (1 + take_profit_pct)
                stop_loss = entry_price * (1 - stop_loss_pct)
                if row['close'] >= take_profit or row['close'] <= stop_loss or row['date'].time() == end_time:
                    position = 0
                    cumulative_pnl += pnl
                    tradebook[-1].update({
                        'Exit Time': row['date'], 'Exit Price': exit_price, 'PNL': pnl,
                        'Exit Reason': 'TP/SL/Close Time', 'Cumulative PNL': cumulative_pnl
                    })

            elif position < 0:  # Short Exit
                take_profit = entry_price * (1 - take_profit_pct)
                stop_loss = entry_price * (1 + stop_loss_pct)
                if row['close'] <= take_profit or row['close'] >= stop_loss or row['date'].time() == end_time:
                    position = 0
                    cumulative_pnl += pnl
                    tradebook[-1].update({
                        'Exit Time': row['date'], 'Exit Price': exit_price, 'PNL': pnl,
                        'Exit Reason': 'TP/SL/Close Time', 'Cumulative PNL': cumulative_pnl
                    })

# ===== Output the Results =====
tradebook_df = pd.DataFrame(tradebook)

# Display and save the tradebook
print(tradebook_df)
tradebook_df.to_csv('/content/drive/MyDrive/1tradebook.csv', index=False)


In [ ]:
!pip install xgboost

In [ ]:
data

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from joblib import Parallel, delayed

# ===== Indicator Calculation =====
def calculate_indicators(df):
    """Compute VWAP, RSI, and Stochastic Oscillator."""
    df['Typical Price'] = (df['high'] + df['low'] + df['close']) / 3
    df['VWAP'] = (df['volume'] * df['Typical Price']).cumsum() / df['volume'].cumsum()

    delta = df['close'].diff()
    gain = delta.where(delta > 0, 0).rolling(31, min_periods=1).mean()
    loss = -delta.where(delta < 0, 0).rolling(31, min_periods=1).mean()
    df['RSI'] = 100 - (100 / (1 + gain / loss))

    df['Low_Min'] = df['low'].rolling(8, min_periods=1).min()
    df['High_Max'] = df['high'].rolling(8, min_periods=1).max()
    df['%K'] = 100 * ((df['close'] - df['Low_Min']) / (df['High_Max'] - df['Low_Min']))
    df['%K'] = df['%K'].rolling(7, min_periods=1).mean()
    df['%D'] = df['%K'].rolling(3, min_periods=1).mean()

    return df.dropna()

# ===== Trading Strategy =====
def apply_trading_strategy(df, capital=100000, stop_loss_pct=0.02, take_profit_pct=0.02, brokerage_pct=0.001):
    """Execute vectorized trading strategy."""
    df['time'] = pd.to_datetime(df['date']).dt.time

    df['Buy'] = (df['RSI'] < 35) & (df['%K'] < 20) & (df['close'] > df['VWAP']) & (df['time'] <= pd.to_datetime('15:00:00').time())
    df['Sell'] = (df['RSI'] > 70) & (df['%K'] > 75) & (df['close'] < df['VWAP']) & (df['time'] <= pd.to_datetime('15:00:00').time())

    trades = []
    position, entry_price, entry_time = 0, 0, None
    cumulative_pnl = 0

    for i, row in df.iterrows():
        if position == 0:
            if row['Buy']:
                position, entry_price, entry_time = capital / row['close'], row['close'], row['date']
            elif row['Sell']:
                position, entry_price, entry_time = -capital / row['close'], row['close'], row['date']

        if position != 0:
            exit_price = row['close']
            pnl = (exit_price - entry_price) * position
            brokerage = brokerage_pct * capital * 2
            final_pnl = pnl - brokerage
            cumulative_pnl += final_pnl

            if ((position > 0 and (exit_price >= entry_price * (1 + take_profit_pct) or exit_price <= entry_price * (1 - stop_loss_pct))) or
                (position < 0 and (exit_price <= entry_price * (1 - take_profit_pct) or exit_price >= entry_price * (1 + stop_loss_pct))) or
                row['time'] >= pd.to_datetime('15:25:00').time() or
                (entry_time and pd.to_datetime(row['date']).date() > pd.to_datetime(entry_time).date())):

                trades.append({
                    'Entry Time': entry_time, 'Exit Time': row['date'],
                    'Entry Price': entry_price, 'Exit Price': exit_price,
                    'Position': 'Long' if position > 0 else 'Short',
                    'PNL': final_pnl,
                    'Cumulative PNL': cumulative_pnl,
                    'Ticker': row['Ticker']
                })
                position = 0

    return pd.DataFrame(trades)

# ===== Machine Learning Model =====
def train_ml_model(data, tradebook):
    """Train ML model to predict profitable trades."""
    features = ['RSI', 'VWAP', '%K', '%D', 'volume', 'close']
    merged_data = pd.merge(data, tradebook, left_on='date', right_on='Entry Time', how='inner')
    merged_data.dropna(subset=features + ['PNL'], inplace=True)

    X, y = merged_data[features], (merged_data['PNL'] > 0).astype(int)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='merror')
    model.fit(X_train, y_train)

    print("Model Accuracy:", accuracy_score(y_test, model.predict(X_test)))
    print(classification_report(y_test, model.predict(X_test)))

    return model

# ===== Live Trading Decision =====
def live_trade_decision(df, model):
    """Use ML model to make live trading decisions."""
    df = calculate_indicators(df)
    features = ['RSI', 'VWAP', '%K', '%D', 'volume', 'close']
    df['Predicted_Profitability'] = model.predict_proba(df[features])[:, 1]
    return df[df['Predicted_Profitability'] > 0.9]

# ===== Backtesting and Live Trading =====
def backtest_and_live_trade(data):
    """Run the entire process: Backtesting, ML training, and live trading."""
    data = calculate_indicators(data)
    tickers = data['Ticker'].unique()

    tradebook_list = Parallel(n_jobs=-1, backend='threading')(
        delayed(process_ticker)(ticker, data) for ticker in tqdm(tickers, desc="Processing Tickers", ncols=100)
    )

    tradebook_df = pd.concat(tradebook_list, ignore_index=True)

    # Train ML Model
    print("Training ML model...")
    ml_model = train_ml_model(data, tradebook_df)

    # Live Trading Decisions
    print("Making live trade decisions...")
    trade_decisions = live_trade_decision(data, ml_model)

    print("High Probability Trades (Buy/Sell Signals):")
    print(trade_decisions)

    return tradebook_df, trade_decisions

# ===== Load Data and Run Backtesting =====
data['date'] = pd.to_datetime(data['date'])

tradebook, trade_decisions = backtest_and_live_trade(data)

# Save Results
tradebook.to_csv('tradebook.csv', index=False)
trade_decisions.to_csv('trade_decisions.csv', index=False)


In [ ]:
tradebook

In [ ]:
import pandas as pd
import numpy as np

# Backtesting Metrics Calculation
def calculate_backtest_metrics(tradebook):
    """
    Calculate key backtesting metrics: Sharpe, Sortino, Drawdown, Win Rate, Profit Factor, etc.
    """
    # Ensure correct data types
    tradebook['Entry Time'] = pd.to_datetime(tradebook['Entry Time'])
    tradebook['Exit Time'] = pd.to_datetime(tradebook['Exit Time'])

    # Sort tradebook by entry time
    tradebook = tradebook.sort_values(by="Entry Time")

    # Cumulative Returns
    tradebook["Cumulative PNL"] = tradebook["PNL"].cumsum()

    # Daily Returns
    tradebook["Daily Returns"] = tradebook["PNL"].groupby('Entry Time') / 100000
    tradebook["Daily Returns"].fillna(0, inplace=True)

    # 1️⃣ Maximum Drawdown (MDD)
    tradebook["Cumulative High"] = tradebook["Cumulative PNL"].cummax()
    tradebook["Drawdown"] = tradebook["Cumulative PNL"] - tradebook["Cumulative High"]
    max_drawdown = tradebook["Drawdown"].min()

    # 2️⃣ Sharpe Ratio
    risk_free_rate = 0  # Assuming 3% annual risk-free rate, daily basis
    sharpe_ratio = (tradebook["Daily Returns"].mean() - risk_free_rate) / tradebook["Daily Returns"].std()

    # 3️⃣ Sortino Ratio (Downside Risk)
    downside_returns = tradebook["Daily Returns"][tradebook["Daily Returns"] < 0]
    sortino_ratio = (tradebook["Daily Returns"].mean() - risk_free_rate) / downside_returns.std()

    # 4️⃣ Calmar Ratio (Return to Drawdown)
    annual_return = tradebook["PNL"].sum() / len(tradebook) * 252  # Assuming daily trades
    calmar_ratio = annual_return / abs(max_drawdown)

    # 5️⃣ Win Rate (% of Profitable Trades)
    win_rate = (len(tradebook["PNL"] > 0)).sum() / len(tradebook)

    # 6️⃣ Profit Factor (Gross Profit / Gross Loss)
    gross_profit = tradebook[tradebook["PNL"] > 0]["PNL"].sum()
    gross_loss = abs(tradebook[tradebook["PNL"] < 0]["PNL"].sum())
    profit_factor = gross_profit / gross_loss if gross_loss != 0 else np.inf

    # 7️⃣ Expectancy (Avg PNL per Trade)
    expectancy = tradebook["PNL"].mean()

    # 8️⃣ Volatility (Standard Deviation of Returns)
    volatility = tradebook["Daily Returns"].std()

    # Results
    results = {
        "Total Trades": len(tradebook),
        "Win Rate": f"{win_rate:.2%}",
        "Profit Factor": round(profit_factor, 2),
        "Expectancy (Avg PNL per Trade)": round(expectancy, 2),
        "Max Drawdown": round(max_drawdown, 2),
        "Sharpe Ratio": round(sharpe_ratio, 2),
        "Sortino Ratio": round(sortino_ratio, 2),
        "Calmar Ratio": round(calmar_ratio, 2),
        "Volatility": round(volatility, 4),
        "Cumulative PNL": round(tradebook["PNL"].sum(), 2),
    }

    return results, tradebook

# Load tradebook CSV (Replace with your actual tradebook)
tradebook = pd.read_csv("tradebook.csv")

# Run the backtest metrics calculation
metrics, updated_tradebook = calculate_backtest_metrics(tradebook)

# Print results
for key, value in metrics.items():
    print(f"{key}: {value}")

# Save updated tradebook with cumulative PNL and drawdowns
updated_tradebook.to_csv("tradebook_with_metrics.csv", index=False)


In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from itertools import product
from joblib import Parallel, delayed  # For parallel processing

# ===== Helper Functions for Indicator Calculations =====
def calculate_indicators(data, rsi_window, stoch_k_window, stoch_d_window):
    """
    Precompute VWAP, RSI, and Stochastic Oscillator for the entire dataset.
    """
    data['Typical Price'] = (data['high'] + data['low'] + data['close']) / 3
    data['Cumulative Vol * Price'] = (data['volume'] * data['Typical Price']).cumsum()
    data['Cumulative Volume'] = data['volume'].cumsum()
    data['VWAP'] = data['Cumulative Vol * Price'] / data['Cumulative Volume']
    del data['Typical Price'], data['Cumulative Vol * Price'], data['Cumulative Volume']  # Free memory

    delta = data['close'].diff()
    gain = delta.clip(lower=0).rolling(window=rsi_window).mean()
    loss = -delta.clip(upper=0).rolling(window=rsi_window).mean()
    data['RSI'] = 100 - (100 / (1 + gain / loss))

    data['Low_Min'] = data['low'].rolling(window=stoch_k_window).min()
    data['High_Max'] = data['high'].rolling(window=stoch_k_window).max()
    data['%K'] = ((data['close'] - data['Low_Min']) / (data['High_Max'] - data['Low_Min'])) * 100
    data['%K'] = data['%K'].rolling(window=stoch_d_window).mean()
    data['%D'] = data['%K'].rolling(window=3).mean()
    data.drop(columns=['Low_Min', 'High_Max'], inplace=True)  # Free memory

    return data


# ===== Backtesting Function =====
def backtest_single_ticker(ticker_data, capital, stop_loss_pct, take_profit_pct,
                           rsi_buy_threshold, rsi_sell_threshold,
                           stoch_k_buy_threshold, stoch_k_sell_threshold,
                           cutoff_time, end_time, brokerage):
    """
    Backtest for a single ticker's data.
    """
    total_pnl = 0
    total_trades = 0
    winning_trades = 0
    position = 0
    entry_price = 0

    ticker_data = ticker_data.dropna(subset=['VWAP', 'RSI', '%K', '%D'])  # Clean data

    for _, row in ticker_data.iterrows():
        # Skip excluded day (default: Friday)
        if row['date'].weekday() == 4:
            continue

        buy_condition = (row['RSI'] < rsi_buy_threshold) and (row['%K'] < stoch_k_buy_threshold) and (row['close'] > row['VWAP'])
        short_condition = (row['RSI'] > rsi_sell_threshold) and (row['%K'] > stoch_k_sell_threshold) and (row['close'] < row['VWAP'])

        # Entry Logic
        if position == 0:
            if buy_condition and row['date'].time() <= cutoff_time:
                position = capital / row['close']
                entry_price = row['close']
            elif short_condition and row['date'].time() <= cutoff_time:
                position = -capital / row['close']
                entry_price = row['close']

        # Exit Logic
        if position != 0:
            exit_price = row['close']
            pnl = (exit_price - entry_price) * position - brokerage * capital

            if position > 0:  # Long Exit
                take_profit = entry_price * (1 + take_profit_pct)
                stop_loss = entry_price * (1 - stop_loss_pct)
                if row['close'] >= take_profit or row['close'] <= stop_loss or row['date'].time() == end_time:
                    total_pnl += pnl
                    total_trades += 1
                    if pnl > 0:
                        winning_trades += 1
                    position = 0

            elif position < 0:  # Short Exit
                take_profit = entry_price * (1 - take_profit_pct)
                stop_loss = entry_price * (1 + stop_loss_pct)
                if row['close'] <= take_profit or row['close'] >= stop_loss or row['date'].time() == end_time:
                    total_pnl += pnl
                    total_trades += 1
                    if pnl > 0:
                        winning_trades += 1
                    position = 0

    win_rate = (winning_trades / total_trades) * 100 if total_trades > 0 else 0
    return total_pnl, win_rate


def backtest_strategy(data, rsi_window, stoch_k_window, stoch_d_window, **params):
    """
    Backtest the strategy for all tickers in the dataset.
    """
    # Precompute indicators for the current set of parameters
    data = calculate_indicators(data.copy(), rsi_window, stoch_k_window, stoch_d_window)

    total_pnl = 0
    total_trades = 0
    winning_trades = 0

    # Iterate through tickers
    for ticker in data['Ticker'].unique():
        ticker_data = data[data['Ticker'] == ticker].copy()
        pnl, win_rate = backtest_single_ticker(ticker_data, **params)
        total_pnl += pnl
        total_trades += total_trades
        winning_trades += winning_trades

    win_rate = (winning_trades / total_trades) * 100 if total_trades > 0 else 0
    return total_pnl, win_rate


# ===== Parallelized Parameter Tuning =====
def run_backtest_for_params(params):
    """
    Wrapper function for parallel backtesting with specific parameters.
    """
    total_pnl, win_rate = backtest_strategy(data=data, **params)
    return {**params, 'Total PnL': total_pnl, 'Win Rate': win_rate}


# Parameter combinations
param_grid = {
    'rsi_window': [31, 41, 51],
    'stoch_k_window': [5, 7, 8, 9],
    'stoch_d_window': [5, 7],
    'capital': [100000],
    'stop_loss_pct': [0.02],
    'take_profit_pct': [0.02],
    'rsi_buy_threshold': [25, 35],
    'rsi_sell_threshold': [70, 75],
    'stoch_k_buy_threshold': [15, 20],
    'stoch_k_sell_threshold': [70, 75],
    'cutoff_time': [pd.to_datetime("11:30").time()],
    'end_time': [pd.to_datetime("15:25").time()],
    'brokerage': [0.001]
}
param_combinations = [dict(zip(param_grid.keys(), values)) for values in product(*param_grid.values())]

# Parallelize the backtesting process
results = Parallel(n_jobs=-1)(delayed(run_backtest_for_params)(params) for params in tqdm(param_combinations))

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv('/content/drive/MyDrive/fine_tuning_results.csv', index=False)
